# Five-Condition Matched-by-Image Analysis (A–E) from 15 AOI Files

This notebook is designed for **your actual data structure**:

- **5 conditions**: A, B, C, D, E
- **3 AOIs**: CTA, Headline, Product
- **one CSV per condition × AOI**
- each CSV contains **20 image rows** (`Img`) and the 8 gaze metrics

So the input is conceptually:

```text
A_CTA.csv        B_CTA.csv        C_CTA.csv        D_CTA.csv        E_CTA.csv
A_Headline.csv   B_Headline.csv   C_Headline.csv   D_Headline.csv   E_Headline.csv
A_Product.csv    B_Product.csv    C_Product.csv    D_Product.csv    E_Product.csv
```

## What this notebook does

For each **AOI × metric**:

1. Reads the five condition files
2. Cleans values like `33.4%`, `2.83s`, `N/A`, `missing`
3. Matches the same `Img` across A/B/C/D/E
4. Runs a **repeated-measures ANOVA** with:
   - **within factor** = condition (A–E)
   - **subject/block** = image (`Img`)
5. Applies **BH-FDR** across the 24 omnibus tests
6. For significant omnibus tests, runs **paired post-hoc comparisons**
   between all condition pairs using **paired t-tests**
7. Applies **BH-FDR** across all post-hoc comparisons
8. Saves clean CSV outputs

## Important note

This analysis is based on **image-level aggregated metrics**, not participant-level data.
So here the repeated unit is the **same advertisement image** across the five conditions.


In [1]:
# Install once if needed:
# %pip install pandas numpy scipy statsmodels

from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.anova import AnovaRM


## 1. Configuration

In [2]:
FILES = {
    "CTA": {
        "A": "data/A_CTA.csv",
        "B": "data/B_CTA.csv",
        "C": "data/C_CTA.csv",
        "D": "data/D_CTA.csv",
        "E": "data/E_CTA.csv",
    },
    "Headline": {
        "A": "data/A_Headline.csv",
        "B": "data/B_Headline.csv",
        "C": "data/C_Headline.csv",
        "D": "data/D_Headline.csv",
        "E": "data/E_Headline.csv",
    },
    "Product": {
        "A": "data/A_Product.csv",
        "B": "data/B_Product.csv",
        "C": "data/C_Product.csv",
        "D": "data/D_Product.csv",
        "E": "data/E_Product.csv",
    },
}
AOI_ORDER = ["CTA", "Headline", "Product"]
CONDITION_ORDER = ["A", "B", "C", "D", "E"]

# Exact input metric names as they appear in your files
INPUT_METRIC_COLUMNS = [
    "VAI%",
    "Avg. TTFF",
    "Fix. Avg. Time Spent",
    "Fixations",
    "Avg. Fixation Duration",
    "Avg. FFD",
    "K-coefficient",
    "Avg. Revisits",
]

# Clean publication-style names
METRIC_RENAME = {
    "VAI%": "VAI (%)",
    "Avg. TTFF": "TTFF (s)",
    "Fix. Avg. Time Spent": "Time spent (s)",
    "Fixations": "Fixations",
    "Avg. Fixation Duration": "Fix. duration (s)",
    "Avg. FFD": "FFD (s)",
    "K-coefficient": "K-coefficient",
    "Avg. Revisits": "Revisits",
}

METRIC_ORDER = [
    "VAI (%)",
    "TTFF (s)",
    "Time spent (s)",
    "Fixations",
    "Fix. duration (s)",
    "FFD (s)",
    "K-coefficient",
    "Revisits",
]

ALPHA = 0.05

OUTPUT_LONG = "A_to_E_long_image_level_data.csv"
OUTPUT_OMNIBUS = "A_to_E_repeated_measures_omnibus.csv"
OUTPUT_POSTHOC = "A_to_E_repeated_measures_posthoc.csv"
OUTPUT_POSTHOC_SIG = "A_to_E_repeated_measures_posthoc_significant.csv"
OUTPUT_SUMMARY = "A_to_E_repeated_measures_summary.csv"


## 2. Cleaning helpers

In [3]:
def clean_text(x):
    if pd.isna(x):
        return x
    return str(x).replace("**", "").strip()


def parse_img(x):
    s = clean_text(x)
    if pd.isna(s):
        return np.nan
    try:
        return int(float(s))
    except Exception:
        return np.nan


def parse_metric_value(x):
    """
    Converts strings like:
    - '33.4%'
    - '2.83s'
    - '0s'
    - 'N/A'
    - 'missing'
    - ''
    into float or NaN.
    """
    if pd.isna(x):
        return np.nan

    s = clean_text(x).lower()

    if s in {"", "n/a", "na", "nan", "missing", "none"}:
        return np.nan

    s = s.replace("%", "")
    s = s.replace("s", "")
    s = s.replace(",", ".")

    try:
        return float(s)
    except Exception:
        return np.nan


def eta_squared_from_anovarm(F, df1, df2):
    """
    Partial eta squared approximation from F and dfs:
    eta_p^2 = (F * df1) / (F * df1 + df2)
    """
    if pd.isna(F) or pd.isna(df1) or pd.isna(df2):
        return np.nan
    denom = F * df1 + df2
    if denom == 0:
        return np.nan
    return (F * df1) / denom


def eta_band(eta):
    if pd.isna(eta):
        return np.nan
    if eta < 0.01:
        return "negligible"
    elif eta < 0.06:
        return "small"
    elif eta < 0.14:
        return "medium"
    else:
        return "large"


## 3. Load all 15 files and convert to long format

In [4]:
all_frames = []

for aoi in AOI_ORDER:
    for condition in CONDITION_ORDER:
        filename = FILES[aoi][condition]
        path = Path(filename)

        if not path.exists():
            raise FileNotFoundError(
                f"Could not find {filename}. "
                "Put the CSVs in the same folder as this notebook "
                "or edit the FILES dictionary."
            )

        df = pd.read_csv(path)
        df.columns = [clean_text(c) for c in df.columns]

        required_cols = ["Img"] + INPUT_METRIC_COLUMNS
        missing_cols = [c for c in required_cols if c not in df.columns]

        if missing_cols:
            raise ValueError(
                f"{filename} is missing columns: {missing_cols}\n"
                f"Available columns: {list(df.columns)}"
            )

        # Keep only the necessary columns
        df = df[required_cols].copy()
        df["Img"] = df["Img"].apply(parse_img)

        # Clean metric values
        for col in INPUT_METRIC_COLUMNS:
            df[col] = df[col].apply(parse_metric_value)

        # Melt to long format
        long_df = df.melt(
            id_vars=["Img"],
            value_vars=INPUT_METRIC_COLUMNS,
            var_name="metric_raw",
            value_name="value"
        )

        long_df["metric"] = long_df["metric_raw"].map(METRIC_RENAME)
        long_df["AOI"] = aoi
        long_df["condition"] = condition

        all_frames.append(
            long_df[["Img", "AOI", "condition", "metric", "value"]]
        )

        print(f"Loaded {filename}: {len(df)} image rows")

long_df = pd.concat(all_frames, ignore_index=True)
long_df = long_df.dropna(subset=["Img"]).copy()

# Order
long_df["AOI"] = pd.Categorical(long_df["AOI"], categories=AOI_ORDER, ordered=True)
long_df["condition"] = pd.Categorical(long_df["condition"], categories=CONDITION_ORDER, ordered=True)
long_df["metric"] = pd.Categorical(long_df["metric"], categories=METRIC_ORDER, ordered=True)

long_df = long_df.sort_values(["AOI", "metric", "Img", "condition"]).reset_index(drop=True)

long_df.to_csv(OUTPUT_LONG, index=False)

print(f"Saved long-format data: {OUTPUT_LONG}")
print("Total rows:", len(long_df))

long_df.head(20)


Loaded data/A_CTA.csv: 20 image rows
Loaded data/B_CTA.csv: 20 image rows
Loaded data/C_CTA.csv: 20 image rows
Loaded data/D_CTA.csv: 20 image rows
Loaded data/E_CTA.csv: 20 image rows
Loaded data/A_Headline.csv: 20 image rows
Loaded data/B_Headline.csv: 20 image rows
Loaded data/C_Headline.csv: 20 image rows
Loaded data/D_Headline.csv: 20 image rows
Loaded data/E_Headline.csv: 20 image rows
Loaded data/A_Product.csv: 20 image rows
Loaded data/B_Product.csv: 20 image rows
Loaded data/C_Product.csv: 20 image rows
Loaded data/D_Product.csv: 20 image rows
Loaded data/E_Product.csv: 20 image rows
Saved long-format data: A_to_E_long_image_level_data.csv
Total rows: 2400


,Img,AOI,condition,metric,value
0,1,CTA,A,VAI (%),0.0
1,1,CTA,B,VAI (%),11.9
2,1,CTA,C,VAI (%),27.8
3,1,CTA,D,VAI (%),11.1
4,1,CTA,E,VAI (%),16.4
5,2,CTA,A,VAI (%),11.4
6,2,CTA,B,VAI (%),21.9
7,2,CTA,C,VAI (%),28.7
8,2,CTA,D,VAI (%),28.4
9,2,CTA,E,VAI (%),22.8


## 4. Quick validation

In [5]:
# Check how many images are available per AOI × metric × condition
coverage = (
    long_df.groupby(["AOI", "metric", "condition"], observed=True)["Img"]
    .nunique()
    .reset_index(name="n_images")
)

display(coverage.head(20))

# Check matching coverage after dropping missing values
for aoi in AOI_ORDER:
    for metric in METRIC_ORDER:
        sub = long_df[(long_df["AOI"] == aoi) & (long_df["metric"] == metric)].copy()

        pivot = sub.pivot(index="Img", columns="condition", values="value")
        complete = pivot.dropna()

        print(
            f"{aoi} / {metric}: "
            f"{len(complete)} complete matched images "
            f"out of {len(pivot)}"
        )


,AOI,metric,condition,n_images
0,CTA,VAI (%),A,20
1,CTA,VAI (%),B,20
2,CTA,VAI (%),C,20
3,CTA,VAI (%),D,20
4,CTA,VAI (%),E,20
5,CTA,TTFF (s),A,20
6,CTA,TTFF (s),B,20
7,CTA,TTFF (s),C,20
8,CTA,TTFF (s),D,20
9,CTA,TTFF (s),E,20


CTA / VAI (%): 17 complete matched images out of 20
CTA / TTFF (s): 16 complete matched images out of 20
CTA / Time spent (s): 17 complete matched images out of 20
CTA / Fixations: 17 complete matched images out of 20
CTA / Fix. duration (s): 16 complete matched images out of 20
CTA / FFD (s): 16 complete matched images out of 20
CTA / K-coefficient: 16 complete matched images out of 20
CTA / Revisits: 17 complete matched images out of 20
Headline / VAI (%): 17 complete matched images out of 20
Headline / TTFF (s): 17 complete matched images out of 20
Headline / Time spent (s): 17 complete matched images out of 20
Headline / Fixations: 17 complete matched images out of 20
Headline / Fix. duration (s): 17 complete matched images out of 20
Headline / FFD (s): 17 complete matched images out of 20
Headline / K-coefficient: 17 complete matched images out of 20
Headline / Revisits: 17 complete matched images out of 20
Product / VAI (%): 17 complete matched images out of 20
Product / TTFF (s)

## 5. Repeated-measures ANOVA for each AOI × metric

For each AOI × metric:
- rows = images
- repeated factor = condition A/B/C/D/E
- subject/block = `Img`

This is appropriate because the **same image** appears in all five conditions.


In [6]:
omnibus_rows = []

for aoi in AOI_ORDER:
    for metric in METRIC_ORDER:
        sub = long_df[
            (long_df["AOI"] == aoi)
            & (long_df["metric"] == metric)
        ].copy()

        # Pivot to keep only images that exist in all 5 conditions
        pivot = sub.pivot(index="Img", columns="condition", values="value")
        complete = pivot.dropna().copy()

        n_images = len(complete)

        if n_images < 2:
            print(f"Skipping {aoi} / {metric}: not enough complete matched images.")
            continue

        # Convert back to long for AnovaRM
        complete_long = (
            complete.reset_index()
                    .melt(id_vars="Img", value_vars=CONDITION_ORDER,
                          var_name="condition", value_name="value")
        )

        complete_long["condition"] = pd.Categorical(
            complete_long["condition"],
            categories=CONDITION_ORDER,
            ordered=True
        )

        # Repeated-measures ANOVA
        try:
            rm = AnovaRM(
                data=complete_long,
                depvar="value",
                subject="Img",
                within=["condition"]
            )
            res = rm.fit()
            anova_table = res.anova_table

            F = float(anova_table.loc["condition", "F Value"])
            df1 = float(anova_table.loc["condition", "Num DF"])
            df2 = float(anova_table.loc["condition", "Den DF"])
            p = float(anova_table.loc["condition", "Pr > F"])

        except Exception as e:
            print(f"Failed {aoi} / {metric}: {e}")
            continue

        eta_sq = eta_squared_from_anovarm(F, df1, df2)

        row = {
            "AOI": aoi,
            "metric": metric,
            "n_images_complete": n_images,
            "F": F,
            "df1": df1,
            "df2": df2,
            "p_omnibus": p,
            "eta_sq": eta_sq,
            "eta_band": eta_band(eta_sq),
        }

        # Add descriptive statistics by condition
        for cond in CONDITION_ORDER:
            vals = complete[cond].astype(float).values
            row[f"mean_{cond}"] = np.mean(vals)
            row[f"sd_{cond}"] = np.std(vals, ddof=1) if len(vals) > 1 else np.nan

        omnibus_rows.append(row)

omnibus = pd.DataFrame(omnibus_rows)

print("Omnibus tests produced:", len(omnibus))
omnibus.head()


Omnibus tests produced: 24


,AOI,metric,n_images_complete,F,df1,df2,p_omnibus,eta_sq,eta_band,mean_A,sd_A,mean_B,sd_B,mean_C,sd_C,mean_D,sd_D,mean_E,sd_E
0,CTA,VAI (%),17,3.283599,4.0,64.0,0.016418,0.170279,large,21.635294,15.756147,26.682353,14.862463,34.482353,13.151722,26.129412,15.839104,28.388235,13.907502
1,CTA,TTFF (s),16,3.237228,4.0,60.0,0.017991,0.177507,large,3.123750,0.986995,2.768125,0.942657,2.516875,0.694624,2.923750,0.823730,3.076875,0.854227
2,CTA,Time spent (s),17,3.623663,4.0,64.0,0.010053,0.184658,large,0.497647,0.280703,0.521765,0.240396,0.637647,0.235466,0.439412,0.221486,0.639412,0.220099
3,CTA,Fixations,17,4.022171,4.0,64.0,0.005680,0.200886,large,17.529412,17.044492,23.294118,19.441466,34.352941,19.332425,22.411765,18.017557,28.470588,17.503991
4,CTA,Fix. duration (s),16,3.182437,4.0,60.0,0.019459,0.175028,large,0.274375,0.067029,0.231875,0.044754,0.232500,0.032965,0.218750,0.032429,0.240000,0.036332


## 6. BH-FDR correction across the 24 omnibus tests

In [7]:
if len(omnibus) == 0:
    raise ValueError("No omnibus results were produced.")

reject, qvals, _, _ = multipletests(
    omnibus["p_omnibus"].values,
    alpha=ALPHA,
    method="fdr_bh"
)

omnibus["q_omnibus_FDR"] = qvals
omnibus["significant_omnibus_FDR"] = reject

omnibus = omnibus.sort_values(["AOI", "metric"]).reset_index(drop=True)

omnibus.to_csv(OUTPUT_OMNIBUS, index=False)
print(f"Saved: {OUTPUT_OMNIBUS}")

display(
    omnibus[
        [
            "AOI",
            "metric",
            "n_images_complete",
            "F",
            "df1",
            "df2",
            "p_omnibus",
            "q_omnibus_FDR",
            "significant_omnibus_FDR",
            "eta_sq",
            "eta_band",
        ]
    ]
)


Saved: A_to_E_repeated_measures_omnibus.csv


,AOI,metric,n_images_complete,F,df1,df2,p_omnibus,q_omnibus_FDR,significant_omnibus_FDR,eta_sq,eta_band
0,CTA,FFD (s),16,2.784858,4.0,60.0,3.442835e-02,0.055085,False,0.156586,large
1,CTA,Fix. duration (s),16,3.182437,4.0,60.0,1.945870e-02,0.037006,True,0.175028,large
2,CTA,Fixations,17,4.022171,4.0,64.0,5.680339e-03,0.017041,True,0.200886,large
3,CTA,K-coefficient,16,0.972355,4.0,60.0,4.294280e-01,0.448099,False,0.060877,medium
4,CTA,Revisits,17,3.145654,4.0,64.0,2.004487e-02,0.037006,True,0.164301,large
5,CTA,TTFF (s),16,3.237228,4.0,60.0,1.799066e-02,0.037006,True,0.177507,large
6,CTA,Time spent (s),17,3.623663,4.0,64.0,1.005300e-02,0.026808,True,0.184658,large
7,CTA,VAI (%),17,3.283599,4.0,64.0,1.641757e-02,0.037006,True,0.170279,large
8,Headline,FFD (s),17,5.455960,4.0,64.0,7.640404e-04,0.003667,True,0.254286,large
9,Headline,Fix. duration (s),17,6.896861,4.0,64.0,1.111486e-04,0.000889,True,0.301214,large


## 7. Paired post-hoc tests for significant omnibus results

For each AOI × metric that survives omnibus FDR, run all 10 pairwise
condition comparisons:

- A vs B
- A vs C
- A vs D
- A vs E
- B vs C
- B vs D
- B vs E
- C vs D
- C vs E
- D vs E

Because the same image is compared across conditions, these are **paired t-tests**.


In [8]:
posthoc_rows = []

for _, omni in omnibus.iterrows():
    if not bool(omni["significant_omnibus_FDR"]):
        continue

    aoi = omni["AOI"]
    metric = omni["metric"]

    sub = long_df[
        (long_df["AOI"] == aoi)
        & (long_df["metric"] == metric)
    ].copy()

    pivot = sub.pivot(index="Img", columns="condition", values="value")
    complete = pivot.dropna().copy()

    for c1, c2 in combinations(CONDITION_ORDER, 2):
        x = complete[c1].astype(float).values
        y = complete[c2].astype(float).values

        if len(x) < 2:
            continue

        t_stat, p_pair = stats.ttest_rel(x, y, nan_policy="omit")

        mean_c1 = np.mean(x)
        mean_c2 = np.mean(y)
        mean_diff = mean_c2 - mean_c1

        posthoc_rows.append({
            "AOI": aoi,
            "metric": metric,
            "group1": c1,
            "group2": c2,
            "n_images_complete": len(complete),
            "mean_group1": mean_c1,
            "mean_group2": mean_c2,
            "mean_difference_group2_minus_group1": mean_diff,
            "t_stat": t_stat,
            "p_posthoc": p_pair,
            "omnibus_F": omni["F"],
            "omnibus_p": omni["p_omnibus"],
            "omnibus_q_FDR": omni["q_omnibus_FDR"],
            "eta_sq_omnibus": omni["eta_sq"],
        })

posthoc = pd.DataFrame(posthoc_rows)

print("Post-hoc rows:", len(posthoc))
posthoc.head()


Post-hoc rows: 140


,AOI,metric,group1,group2,n_images_complete,mean_group1,mean_group2,mean_difference_group2_minus_group1,t_stat,p_posthoc,omnibus_F,omnibus_p,omnibus_q_FDR,eta_sq_omnibus
0,CTA,Fix. duration (s),A,B,16,0.274375,0.231875,-0.042500,2.633629,0.018796,3.182437,0.019459,0.037006,0.175028
1,CTA,Fix. duration (s),A,C,16,0.274375,0.232500,-0.041875,2.196308,0.044206,3.182437,0.019459,0.037006,0.175028
2,CTA,Fix. duration (s),A,D,16,0.274375,0.218750,-0.055625,2.586583,0.020643,3.182437,0.019459,0.037006,0.175028
3,CTA,Fix. duration (s),A,E,16,0.274375,0.240000,-0.034375,1.563619,0.138755,3.182437,0.019459,0.037006,0.175028
4,CTA,Fix. duration (s),B,C,16,0.231875,0.232500,0.000625,-0.037105,0.970891,3.182437,0.019459,0.037006,0.175028


## 8. BH-FDR correction across all post-hoc tests

In [9]:
if len(posthoc) > 0:
    reject_ph, q_ph, _, _ = multipletests(
        posthoc["p_posthoc"].values,
        alpha=ALPHA,
        method="fdr_bh"
    )

    posthoc["q_posthoc_global_FDR"] = q_ph
    posthoc["significant_posthoc_global_FDR"] = reject_ph

    posthoc = posthoc.sort_values(
        ["AOI", "metric", "q_posthoc_global_FDR", "group1", "group2"]
    ).reset_index(drop=True)

    posthoc.to_csv(OUTPUT_POSTHOC, index=False)
    print(f"Saved: {OUTPUT_POSTHOC}")

    sig_posthoc = posthoc[posthoc["significant_posthoc_global_FDR"]].copy()
    sig_posthoc.to_csv(OUTPUT_POSTHOC_SIG, index=False)
    print(f"Saved: {OUTPUT_POSTHOC_SIG}")

    display(
        sig_posthoc[
            [
                "AOI",
                "metric",
                "group1",
                "group2",
                "mean_group1",
                "mean_group2",
                "mean_difference_group2_minus_group1",
                "t_stat",
                "p_posthoc",
                "q_posthoc_global_FDR",
                "eta_sq_omnibus",
            ]
        ]
    )
else:
    print("No post-hoc results to correct.")


Saved: A_to_E_repeated_measures_posthoc.csv
Saved: A_to_E_repeated_measures_posthoc_significant.csv


,AOI,metric,group1,group2,mean_group1,mean_group2,mean_difference_group2_minus_group1,t_stat,p_posthoc,q_posthoc_global_FDR,eta_sq_omnibus
10,CTA,Fixations,A,C,17.529412,34.352941,16.823529,-3.559392,0.002614,0.022874,0.200886
20,CTA,Revisits,A,C,0.248235,0.548235,0.300000,-3.880361,0.001328,0.014297,0.164301
30,CTA,TTFF (s),C,D,2.516875,2.923750,0.406875,-3.503367,0.003201,0.025557,0.177507
31,CTA,TTFF (s),A,C,3.123750,2.516875,-0.606875,2.973731,0.009466,0.045369,0.177507
40,CTA,Time spent (s),C,D,0.637647,0.439412,-0.198235,3.325993,0.004278,0.029949,0.184658
50,CTA,VAI (%),A,C,21.635294,34.482353,12.847059,-3.160952,0.006054,0.034510,0.170279
60,Headline,FFD (s),A,B,0.277647,0.220588,-0.057059,3.451105,0.003286,0.025557,0.254286
61,Headline,FFD (s),A,C,0.277647,0.221176,-0.056471,3.143320,0.006283,0.034510,0.254286
62,Headline,FFD (s),A,E,0.277647,0.224706,-0.052941,3.024249,0.008061,0.041695,0.254286
63,Headline,FFD (s),A,D,0.277647,0.225882,-0.051765,2.934352,0.009722,0.045369,0.254286


## 9. Summary table

In [10]:
summary = omnibus[
    [
        "AOI",
        "metric",
        "n_images_complete",
        "F",
        "df1",
        "df2",
        "p_omnibus",
        "q_omnibus_FDR",
        "significant_omnibus_FDR",
        "eta_sq",
        "eta_band",
    ]
].copy()

summary.to_csv(OUTPUT_SUMMARY, index=False)
print(f"Saved: {OUTPUT_SUMMARY}")

summary


Saved: A_to_E_repeated_measures_summary.csv


,AOI,metric,n_images_complete,F,df1,df2,p_omnibus,q_omnibus_FDR,significant_omnibus_FDR,eta_sq,eta_band
0,CTA,FFD (s),16,2.784858,4.0,60.0,3.442835e-02,0.055085,False,0.156586,large
1,CTA,Fix. duration (s),16,3.182437,4.0,60.0,1.945870e-02,0.037006,True,0.175028,large
2,CTA,Fixations,17,4.022171,4.0,64.0,5.680339e-03,0.017041,True,0.200886,large
3,CTA,K-coefficient,16,0.972355,4.0,60.0,4.294280e-01,0.448099,False,0.060877,medium
4,CTA,Revisits,17,3.145654,4.0,64.0,2.004487e-02,0.037006,True,0.164301,large
5,CTA,TTFF (s),16,3.237228,4.0,60.0,1.799066e-02,0.037006,True,0.177507,large
6,CTA,Time spent (s),17,3.623663,4.0,64.0,1.005300e-02,0.026808,True,0.184658,large
7,CTA,VAI (%),17,3.283599,4.0,64.0,1.641757e-02,0.037006,True,0.170279,large
8,Headline,FFD (s),17,5.455960,4.0,64.0,7.640404e-04,0.003667,True,0.254286,large
9,Headline,Fix. duration (s),17,6.896861,4.0,64.0,1.111486e-04,0.000889,True,0.301214,large


## 10. Specifically inspect whether C differs from B, D, and E

In [11]:
if len(posthoc) > 0:
    c_vs_others = posthoc[
        (
            (posthoc["group1"] == "C") & (posthoc["group2"].isin(["B", "D", "E"]))
        )
        |
        (
            (posthoc["group2"] == "C") & (posthoc["group1"].isin(["B", "D", "E"]))
        )
    ].copy()

    display(
        c_vs_others[
            [
                "AOI",
                "metric",
                "group1",
                "group2",
                "mean_group1",
                "mean_group2",
                "mean_difference_group2_minus_group1",
                "p_posthoc",
                "q_posthoc_global_FDR",
                "significant_posthoc_global_FDR",
            ]
        ].sort_values(["AOI", "metric", "q_posthoc_global_FDR"])
    )
else:
    print("No post-hoc results available.")


,AOI,metric,group1,group2,mean_group1,mean_group2,mean_difference_group2_minus_group1,p_posthoc,q_posthoc_global_FDR,significant_posthoc_global_FDR
5,CTA,Fix. duration (s),C,D,0.232500,0.218750,-0.013750,0.240483,0.391483,False
8,CTA,Fix. duration (s),C,E,0.232500,0.240000,0.007500,0.609135,0.735163,False
9,CTA,Fix. duration (s),B,C,0.231875,0.232500,0.000625,0.970891,0.979363,False
11,CTA,Fixations,C,D,34.352941,22.411765,-11.941176,0.011466,0.050165,False
12,CTA,Fixations,B,C,23.294118,34.352941,11.058824,0.018785,0.073095,False
17,CTA,Fixations,C,E,34.352941,28.470588,-5.882353,0.277099,0.431043,False
21,CTA,Revisits,C,D,0.548235,0.344706,-0.203529,0.020469,0.076052,False
22,CTA,Revisits,B,C,0.352353,0.548235,0.195882,0.041087,0.134071,False
26,CTA,Revisits,C,E,0.548235,0.420588,-0.127647,0.306511,0.461415,False
30,CTA,TTFF (s),C,D,2.516875,2.923750,0.406875,0.003201,0.025557,True


## How to interpret the outputs

### `A_to_E_repeated_measures_omnibus.csv`
This is the main omnibus result.
If:

```text
significant_omnibus_FDR = True
```

then for that AOI × metric, **at least one condition differs from another** after BH-FDR.

### `A_to_E_repeated_measures_posthoc.csv`
This gives all pairwise paired comparisons for the omnibus-significant AOI × metric combinations.

### `A_to_E_repeated_measures_posthoc_significant.csv`
This is the easiest post-hoc file to inspect.
It contains only pairwise comparisons that survive the global post-hoc FDR correction.

### Practical question for your project
Look especially at:

- `A` vs `C`
- `B` vs `C`
- `C` vs `D`
- `C` vs `E`

especially for CTA metrics such as:

- VAI (%)
- TTFF (s)
- Time spent (s)
- Fixations
- Revisits

This will tell you whether condition **C** is really different from the other edited conditions when the same images are compared across A–E.
